# Week 3 briefing: AlphaEarth RES / NRES

Building-scale AlphaEarth still beats GHSL (about 0.75 vs 0.67 macro-F1). At 10 m pixels the same signal is weaker (binary around 0.56). Our best pixel recipe so far adds an empty class and a soft-fraction head with decode τ=0.15: 0.52 OOF macro-F1, ahead in 9 of 10 metros. A 3×3 Conv neighborhood scored about 0.35 and did not help.

Setup: leave-one-metro-out (LORO), 10 US metros, Track-A Overture labels, pure pixels (mixed dropped) unless noted. Figures below are already under `outputs/`; you do not need to retrain to read this notebook.


## 0. The ladder

Ranking of every Week 3 setup we ran. Building AE is on top. Soft-fraction 3-class is the best pixel line. The 3×3 CNN is last.

![Building-scale AE still leads; soft-fraction pixels beat hard labels; 3×3 CNN does not](../../outputs/week3_model_comparison/figures/08_week3_f1_ladder.png)

![Only building-scale AE beats our best 10 m recipe](../../outputs/week3_model_comparison/figures/09_gap_vs_best_pixel_recipe.png)


## 1. Building scale: AE CNN beats GHSL

On footprints (not pixels), the AlphaEarth bbox CNN clears GHSL by about 0.07 macro-F1.

![On buildings, Polygon AE CNN beats GHSL](../../outputs/week3_model_comparison/figures/01_building_ae_beats_ghsl.png)

![Poly − GHSL by metro](../../outputs/week3_model_comparison/figures/04_poly_minus_ghsl_by_city.png)


## 2. How we score GHSL at the pixel level

GHS-BUILT-C is a 10 m raster. Pixel F1 is not every cell on Earth; it is cells whose center falls inside a Track-A footprint, painted onto the exact GHSL grid, then compared to the GHSL code at that cell.

![GHSL pixel eval steps](../../outputs/validation/figures/ghsl_pixel_eval_schematic.png)

Boston chip: GHSL map + footprints, GT paint, and the eval mask.

![GHSL pixel eval real chip](../../outputs/validation/figures/ghsl_pixel_eval_real_chip.png)

On those shared cells: GT vs GHSL vs agree/disagree.

![GHSL pixel agree/disagree](../../outputs/validation/figures/ghsl_pixel_eval_agree_disagree.png)

Config text says ≥50% footprint coverage; the burn uses center-in-polygon (binary). On the Boston audit chip (242 eval cells), 6 (2.48%) have true area &lt; 0.5; agree 79 · disagree 153 · GHSL uncovered 10.

![Center proxy vs true area fraction](../../outputs/validation/figures/ghsl_pixel_eval_center_vs_area.png)

Code: `validation/pixel_eval.py`, `scripts/plot_ghsl_pixel_eval_method.py`. Audit: `outputs/validation/ghsl_pixel_eval_audit.json`.


## 3. Moving to 10 m pixels costs about 0.18 F1

Same AE embeddings, harder unit. Building scores and pixel scores measure different things. The gap still matters if the product needs grid cells.

![Building to pixel costs ~0.18 F1 for AE](../../outputs/week3_model_comparison/figures/03_unit_drop_building_to_pixel.png)

![Unit scoreboards](../../outputs/week3_model_comparison/figures/07_unit_scoreboards.png)


## 4. How 10 m pixels meet Overture

Two different constructions sit in the same table. That is what makes the empty class look small.

### Visual: fractions inside one cell

![Coverage fractions cartoon: edge clip, mostly covered, true empty](../../outputs/week3_pixel_3class/figures/pixel_overlap_schematic.png)

### Visual: two sampling paths

![Building-overlapped pixels vs AOI-sampled empties](../../outputs/week3_pixel_3class/figures/pixel_sampling_two_paths.png)

### Visual: real Overture overlays (Boston)

![Real Boston examples of partial RES, covered NRES, and AOI empty](../../outputs/week3_pixel_3class/figures/pixel_overlap_real_examples.png)

### Building-overlapped pixels (RES / NRES / mixed)

Start from Track-A Overture footprints (`resi_def_v1` class GT) and the AlphaEarth 10 m cells that already appear in each building's enclosing bbox grid (`datasets/building_bbox_grid/`). Keep each unique 10 m cell that intersects at least one GT footprint.

For that cell:

```text
frac_res  = area(pixel ∩ union(RES footprints))  / pixel_area
frac_nres = area(pixel ∩ union(NRES footprints)) / pixel_area
frac_empty = clip(1 - frac_res - frac_nres, 0, 1)
```

Hard labels are presence-based (any overlap counts), not the cell τ=0.7 rule from the residential definition doc:

| Label | Rule |
|---|---|
| residential | `frac_res > 0` and `frac_nres == 0` |
| non_residential | `frac_nres > 0` and `frac_res == 0` |
| mixed | both > 0 (dropped from train) |

So yes: every RES / NRES / mixed pixel is next to (actually intersecting) a labeled polygon. A cell with only 2% building cover is still labeled RES or NRES. That is where the "mostly empty band" in the next section comes from: footprint edge clips, not true empties.

About 283k such pixels in v1; NRES dominates (~253k) because non-residential footprints cover more of the sampled grids.

### Empty pixels (AOI sample, not building neighbors)

True empties are not taken from cells near polygons. We uniform-sample candidate 10 m centers inside each metro AOI box, snap them to the AE grid, and keep a cell only if it intersects zero Track-A footprints. Embeddings are then read from the AlphaEarth COG.

Cap per metro: `n_empty ≈ n_residential` (seeded reject sample). That is why the empty class looks small next to NRES (~29k empty vs ~30k RES vs ~253k NRES in v2). It is a balanced draw against residential, not a census of every vacant cell in the AOI.

| Piece | Near polygons? | How many |
|---|---|---|
| RES / NRES / mixed | Yes (must intersect a Track-A footprint) | All unique overlapping AE cells in the bbox-grid cache |
| empty | No (AOI-wide, reject if any footprint hit) | Capped to ~n_residential per metro |

Scripts: `scripts/build_week3_pixel_gt.py`, `scripts/build_week3_pixel_gt_with_empty.py`. Manifest: `datasets/week3_pixel_gt_v2/manifest.json`.


## 5. Pixel binary: LR and soft-regression nearly tied

With pure RES/NRES and SMOTE, ridge / soft-reg lands at 0.57 and logistic at 0.56. Both beat pixel GHSL in most metros. Neither gets close to building AE.

![On pixels, AE LR and AE regression are nearly tied](../../outputs/week3_model_comparison/figures/02_pixel_lr_vs_frac_vs_ghsl.png)

![LR − frac by city](../../outputs/week3_model_comparison/figures/06_lr_minus_frac_by_city.png)

![Pixel LR − GHSL by city](../../outputs/week3_model_comparison/figures/05_pixel_lr_minus_ghsl_by_city.png)


## 5b. LLD: pixel-level models

Shared Track-A × AlphaEarth cells, then five heads (binary LR / binary reg / 3-class clf / soft-frac+τ / 3×3 NbCNN).

![LLD pixel-level models](../../outputs/week3_model_comparison/figures/lld_pixel_level_models.png)


## 6. Why pixels are hard: imbalance and edge cells

NRES is about 81% of the pixel set. In the band where empty > 90% and each building class stays under 10%, only about 42% of cells have zero footprint overlap. The other 58% are partial buildings clipping the cell. Treating everything under 5% building share as empty would wipe those out.

![Label counts (v2 with empties)](../../datasets/week3_pixel_gt_v2/figures/label_counts.png)

![Label by region](../../datasets/week3_pixel_gt_v2/figures/label_by_region.png)

![Mostly-empty band: true empty vs partial by metro](../../outputs/week3_pixel_3class/figures/mostly_empty_band_by_region.png)

![GT cells with tiny building share, Overture overlays](../../outputs/week3_pixel_3class/figures/mostly_empty_band_overture_samples.png)

![Partial-overlap building-share histogram](../../outputs/week3_pixel_3class/figures/mostly_empty_band_partial_share_hist.png)


## 7. Three-class (RES / NRES / empty): soft fractions win

We train on soft `frac_res / frac_nres / frac_empty` and decode with building share < 0.15 as empty. That head beats hard-label classification on macro-F1 (0.52 vs 0.46 OOF) and wins in 9 of 10 metros. The classifier still recalls empty better (0.62 vs 0.45), so there is a precision/recall tradeoff.

![Classifier vs regression head by metro (3-class)](../../outputs/week3_pixel_3class/figures/loro_f1_clf_vs_reg.png)

![Regression beats classifier in 9/10 metros](../../outputs/week3_pixel_3class/figures/delta_reg_minus_clf_by_city.png)

![OOF confusion: clf vs soft-reg decode](../../outputs/week3_pixel_3class/figures/oof_confusion_clf_vs_reg.png)


## 8. Neighborhood experiment: 3×3 Conv did not help

Same 3-class labels, but features are a centered 3×3 AE window through a Conv encoder (masked GAP). OOF macro-F1 falls to about 0.35 / 0.33, below single-pixel soft-reg at 0.52.

![3×3 CNN clf vs reg by metro](../../outputs/week3_pixel_neighborhood_k3/figures/loro_f1_clf_vs_reg.png)

![Reg − clf gaps (3×3)](../../outputs/week3_pixel_neighborhood_k3/figures/delta_reg_minus_clf_by_city.png)

![3×3 OOF confusion](../../outputs/week3_pixel_neighborhood_k3/figures/oof_confusion_clf_vs_reg.png)

### Embedding view (per metro)

UMAP of the OOF 128-D pooled CNN features. Within metros, RES and NRES still overlap a lot.

![UMAP clf by label per region](../../outputs/week3_pixel_neighborhood_k3/figures/umap_clf_by_label_per_region.png)

![UMAP reg by label per region](../../outputs/week3_pixel_neighborhood_k3/figures/umap_reg_by_label_per_region.png)


## 9. Takeaways

| Decision | Evidence |
|---|---|
| Prefer building-scale AE when footprints exist | 0.75 macro-F1 vs GHSL 0.67 |
| Expect a large drop at 10 m grids | about -0.18 F1 vs building AE |
| For pixel 3-class, use soft-fraction + τ≈0.15 | 0.52 OOF; 9/10 metros beat hard clf |
| Do not treat <5-10% building as empty in GT | 58% of the mostly-empty band are real edge clips |
| 3×3 Conv is not the next lever yet | 0.35 F1; weak UMAP separation |

Next steps we are considering:

1. Learn a presence head instead of a fixed τ (empty recall still lags the classifier).
2. Keep single-pixel soft-reg as the pixel baseline for demos.
3. Revisit context only with a stronger encoder or multi-scale features, not this tiny CNN.

Related notebooks: `01` data, `02` pixel LR, `03` fraction reg, `05` 3-class, `06` neighborhood. Summaries live under `outputs/week3_*/summary.json`.


## Appendix: load live numbers


In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path('.').resolve()
for _ in range(3):
    if (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent

paths = {
    'four-model comparison': ROOT / 'outputs/week3_model_comparison/summary.json',
    '3-class single-pixel': ROOT / 'outputs/week3_pixel_3class/summary.json',
    '3×3 neighborhood CNN': ROOT / 'outputs/week3_pixel_neighborhood_k3/summary.json',
}
for name, p in paths.items():
    d = json.loads(p.read_text())
    display(Markdown(f'### {name}'))
    # keep it short for the room
    keep = {k: d[k] for k in d if any(s in k for s in ('f1', 'building_share', 'reg_beats', 'n_pixels', 'k', 'encoder', 'claims', 'caveat'))}
    print(json.dumps(keep, indent=2)[:2000])


### four-model comparison

{
  "claims": [
    "On buildings, Polygon AE CNN beats GHSL by ~0.07 macro-F1.",
    "On 10 m pixels (pure+SMOTE), regression (0.57) edges LR (0.56).",
    "Switching the unit from building to pixel costs ~0.18 F1 for AE."
  ],
  "caveat": "Building and pixel F1 are different tasks; within-unit panels are the fair comparisons. Pixel models: pure only + SMOTE.",
  "week2_binary_means": {
    "lr_centroid_macro_f1_mean": 0.6958862087957314,
    "cnn_bbox_macro_f1_mean": 0.7463500019458852,
    "delta_mean": 0.05046379315015383
  }
}


### 3-class single-pixel

{
  "n_pixels": 312086,
  "building_share_max": 0.15,
  "clf_loro_macro_f1_mean": 0.4462323300607064,
  "clf_loro_macro_f1_std": 0.05435038356839486,
  "reg_loro_macro_f1_mean": 0.5030133981804403,
  "reg_loro_macro_f1_std": 0.0614860504776796,
  "oof_clf_macro_f1": 0.4572393799964856,
  "oof_reg_macro_f1": 0.5211295030279147,
  "reg_beats_clf_metros": 9
}


### 3×3 neighborhood CNN

{
  "k": 3,
  "encoder": "NbCNN",
  "n_pixels": 308675,
  "building_share_max": 0.15,
  "clf_loro_macro_f1_mean": 0.3454445308190508,
  "clf_loro_macro_f1_std": 0.12599842589481983,
  "reg_loro_macro_f1_mean": 0.32422182570598745,
  "reg_loro_macro_f1_std": 0.10115118262265518,
  "oof_clf_macro_f1": 0.3533789223291353,
  "oof_reg_macro_f1": 0.33404010130942874,
  "reg_beats_clf_metros": 4
}
